# 3. Graph-EVT: одноканальная POT-детекция

**Цель:** применить `GraphEVTPipeline` к одномерному ряду, оценить POT/GPD-порог на начальном baseline и проверить наличие первого устойчивого превышения. Это детерминированный анализ библиотеки, не LLM-исследование. Для одного канала библиотека не строит граф или рейтинг источников и не выдаёт отдельные подгонки хвостов либо declustering-таблицы.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import json
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/fractal_extreme_series.csv").exists())
DATA = ROOT / "data/fractal_extreme_series.csv"
SEED = 42
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
assert np.isfinite(x).all() and x.ndim == 1


In [ ]:
BASELINE_SIZE = int(0.75 * len(x))
evt_config = EVTConfig(
    baseline_size=BASELINE_SIZE,
    tail_quantile=0.95,
    alarm_probability=0.999,
    persistence=3,
)
pipeline = GraphEVTPipeline(
    evt_config,
    graph=GraphConfig(random_state=SEED),
    input_config=InputConfig(missing="error"),
)
result = pipeline.run_detailed(x)
detection = result.detection

pd.DataFrame([{
    "observations": result.input_profile.n_time,
    "baseline_size": BASELINE_SIZE,
    "alarm_threshold": detection.alarm_threshold,
    "detected": detection.detected,
    "time_index": detection.time_index,
    "graph_available": result.graph is not None,
}])


In [ ]:
artifact_dir = ROOT / "results"
artifact_dir.mkdir(exist_ok=True)

indicator_frame = pd.DataFrame({
    "time": df["time"],
    "value": x,
    "indicator": detection.indicator,
    "above_alarm_threshold": detection.indicator > detection.alarm_threshold,
    "is_baseline": np.arange(len(x)) < BASELINE_SIZE,
})
indicator_frame.to_csv(artifact_dir / "03_evt_indicator.csv", index=False)

provenance = {
    "library": "graph-evt-agent",
    "library_version": version("graph-evt-agent"),
    "input_file": DATA.name,
    "observations": result.input_profile.n_time,
    "seed": SEED,
    "configuration": {
        "baseline_size": BASELINE_SIZE,
        "tail_quantile": evt_config.tail_quantile,
        "alarm_probability": evt_config.alarm_probability,
        "persistence": evt_config.persistence,
    },
    "detection": {
        "detected": detection.detected,
        "time_index": detection.time_index,
        "alarm_threshold": float(detection.alarm_threshold),
        "location": detection.location.tolist(),
        "scale": detection.scale.tolist(),
    },
    "graph_available": result.graph is not None,
    "ranking_available": result.ranking is not None,
}
with (artifact_dir / "03_evt_provenance.json").open("w", encoding="utf-8") as file:
    json.dump(provenance, file, indent=2)

provenance


Оценка использует робастные центр и масштаб первых 75% ряда, двусторонний score абсолютного отклонения и POT/GPD-калибровку alarm-порога. Это baseline-допущение нужно проверять: синтетический ряд не содержит опубликованной разметки импульсов. При текущих настройках устойчивое превышение не найдено; это не доказывает отсутствие экстремумов.

Проверьте чувствительность к размеру baseline, `tail_quantile`, `alarm_probability` и `persistence`. Знак отклонения объединён в один score; эта версия библиотеки не строит отдельные положительный/отрицательный tail fits и не выполняет declustering. Для одного канала граф и ranking закономерно отсутствуют.

**Самопроверка:** как выбор baseline влияет на порог? Что теряется при объединении двух направлений отклонения? Почему для одного канала нельзя локализовать источник?
